# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

*Loads the decision-point frame from ML-04: one row per content item, features from Feb-Apr 2026, label `y_decline_may` from May 2026. If the cached file is missing, the cell rebuilds it from the warehouse (needs `HF_TOKEN`). June 2026 is never loaded.*

In [1]:
# Setup | load the decision-point frame; build it from the warehouse only if it is not cached (features Feb-Apr 2026, label May 2026; June is never loaded)
import gc, json, os
import numpy as np, pandas as pd

FRAME = "work/outputs/frame_dp_2026-05-01.parquet"
os.makedirs("work/outputs", exist_ok=True)

def hf_token():
    try:
        from google.colab import userdata          # Colab: Secrets panel
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")          # local run: environment variable

if os.path.exists(FRAME):
    print("Using the cached frame:", FRAME)
else:
    # ML-04 v2 | Decision-point frame: features Feb-Apr 2026, label May 2026. June stays sealed (never loaded).
    import gc, json, os
    import numpy as np, pandas as pd

    HF = {"token": hf_token()}
    HF_BASE = "hf://datasets/FlyRank/internship-warehouse"
    FEAT_MONTHS, LABEL_MONTH = ["2026-02", "2026-03", "2026-04"], "2026-05"
    MIN_IMPR_WINDOW, MIN_CLICKS_APR, DROP_RATIO = 1000, 10, 0.80      # contract thresholds: frozen before any model
    KEYS = ["client_hash_id", "content_hash_id"]
    COLS = ["report_date", *KEYS, "gsc_clicks", "gsc_impressions", "gsc_avg_position"]
    assert LABEL_MONTH != "2026-06" and "2026-06" not in FEAT_MONTHS, "June is the sealed test month"

    def month_agg(m):
        """One partition -> one row per (client, content). Loads only needed columns; frees RAM right after."""
        path = f"{HF_BASE}/fact_content_daily_performance/month={m}/"
        try:
            d = pd.read_parquet(path, columns=COLS, storage_options=HF)
        except Exception:
            from huggingface_hub import HfFileSystem
            import pyarrow.parquet as pq
            fs = HfFileSystem(token=HF["token"])
            f = fs.ls(f"datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}", detail=False)[0]
            print("Actual schema:", pq.read_schema(f, filesystem=fs).names); raise
        dates = pd.to_datetime(d["report_date"])
        win = (dates.min().date(), dates.max().date(), len(d))
        impr = d["gsc_impressions"].fillna(0)
        pos = d["gsc_avg_position"].where(d["gsc_avg_position"] > 0)            # 0/negative = no data, not rank 0
        d = d.assign(clicks=d["gsc_clicks"].fillna(0), impr=impr,
                     pos_x_impr=(pos * impr).fillna(0), impr_pos=impr.where(pos.notna(), 0),
                     active=(impr > 0).astype("int16"))
        g = d.groupby(KEYS, observed=True).agg(clicks=("clicks", "sum"), impr=("impr", "sum"),
                                               pos_x_impr=("pos_x_impr", "sum"), impr_pos=("impr_pos", "sum"),
                                               active_days=("active", "sum")).reset_index()
        del d; gc.collect()
        return g, win

    # 1) Aggregate each month separately (daily grain -> content grain), check windows land inside their month
    frames, windows = {}, {}
    for m in FEAT_MONTHS + [LABEL_MONTH]:
        frames[m], windows[m] = month_agg(m)
        assert str(windows[m][0]).startswith(m) and str(windows[m][1]).startswith(m), f"{m} partition has dates outside its month"
        print(f"{m}: daily rows={windows[m][2]:,} | {windows[m][0]} -> {windows[m][1]} | contents={len(frames[m]):,}")

    # 2) Feature frame from Feb-Apr ONLY (nine features, all knowable on 2026-05-01)
    f = None
    for m in FEAT_MONTHS:
        part = frames[m].rename(columns={c: f"{c}_{m}" for c in ["clicks", "impr", "pos_x_impr", "impr_pos", "active_days"]})
        f = part if f is None else f.merge(part, on=KEYS, how="outer")
    num = [c for c in f.columns if c not in KEYS]
    f[num] = f[num].fillna(0)                                             # absent row in a month = no search rows that month
    S = lambda p: sum(f[f"{p}_{m}"] for m in FEAT_MONTHS)
    f["impressions_total"], f["clicks_total"] = S("impr"), S("clicks")
    f["impressions_apr"], f["clicks_apr"], f["clicks_feb"] = f["impr_2026-04"], f["clicks_2026-04"], f["clicks_2026-02"]
    f["momentum_apr_vs_feb"] = np.log1p(f["clicks_apr"]) - np.log1p(f["clicks_feb"])
    f["ctr"] = np.where(f["impressions_total"] > 0, f["clicks_total"] / f["impressions_total"], np.nan)
    f["weighted_position"] = np.where(S("impr_pos") > 0, S("pos_x_impr") / S("impr_pos").replace(0, np.nan), np.nan)  # sum(pos*impr)/sum(impr), not a mean of daily means
    f["active_days"] = S("active_days")
    FEATURES = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb",
                "momentum_apr_vs_feb", "ctr", "weighted_position", "active_days"]
    f = f[KEYS + FEATURES]

    # 3) Label from May ONLY: decline = May clicks < 80% of April clicks. Label ingredients are never features.
    may = frames[LABEL_MONTH][KEYS + ["clicks"]].rename(columns={"clicks": "clicks_may"})
    n_before_eligibility = len(f)
    f = f[(f["impressions_total"] >= MIN_IMPR_WINDOW) & (f["clicks_apr"] >= MIN_CLICKS_APR)]   # eligibility, decided at the decision point
    f = f.merge(may, on=KEYS, how="left")
    missing_in_may = int(f["clicks_may"].isna().sum())
    f["clicks_may"] = f["clicks_may"].fillna(0)                           # not seen in May -> 0 clicks -> counts as decline (stated in contract)
    f["y_decline_may"] = (f["clicks_may"] < DROP_RATIO * f["clicks_apr"]).astype(int)
    LABEL_INGREDIENTS = ["clicks_may"]                                    # kept ONLY to build/audit the label
    frame = f.reset_index(drop=True)

    # 4) Contract checks (queries, not claims)
    assert not frame.duplicated(KEYS).any(), "grain broken: (client, content) must be unique"
    assert not set(LABEL_INGREDIENTS) & set(FEATURES), "label ingredient leaked into features"
    assert set(KEYS).isdisjoint(FEATURES), "IDs are grouping keys, never features"
    share = frame["client_hash_id"].value_counts(normalize=True)
    base_rate = float(frame["y_decline_may"].mean())
    print(f"\nBefore eligibility: {n_before_eligibility:,} | Eligible rows: {len(frame):,} | Clients: {frame['client_hash_id'].nunique()}")
    print(f"Base rate (decline in May): {base_rate:.3f} | Not seen in May (counted as decline): {missing_in_may:,}")
    print(f"Largest client share of rows: {share.iloc[0]:.1%} | Top-5 clients: {share.head(5).sum():.1%}")
    print("Per-client base rate spread (min / median / max, clients with >=30 rows):")
    pc = frame.groupby("client_hash_id")["y_decline_may"].agg(["mean", "size"]); pc = pc[pc["size"] >= 30]["mean"]
    print(f"  {pc.min():.2f} / {pc.median():.2f} / {pc.max():.2f}  over {len(pc)} clients")
    print("Missing in features (share):"); print(frame[FEATURES].isna().mean().round(3).to_string())

    # 5) Receipt (commit this JSON), cache the frame locally (do NOT commit the parquet)
    os.makedirs("work/outputs", exist_ok=True)
    frame.to_parquet("work/outputs/frame_dp_2026-05-01.parquet", index=False)
    json.dump({"decision_point": "2026-05-01", "feature_months": FEAT_MONTHS, "label_month": LABEL_MONTH,
               "min_impr_window": MIN_IMPR_WINDOW, "min_clicks_apr": MIN_CLICKS_APR, "drop_ratio": DROP_RATIO,
               "rows_before_eligibility": int(n_before_eligibility), "rows": int(len(frame)),
               "clients": int(frame["client_hash_id"].nunique()), "base_rate": round(base_rate, 4),
               "not_seen_in_may": missing_in_may, "features": FEATURES, "label": "y_decline_may",
               "group_key": "client_hash_id", "sealed_month": "2026-06", "release": "v20260703"},
              open("work/outputs/data_contract_v2.json", "w"), indent=2)
    print("\nSaved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)")

2026-02: daily rows=7,355,108 | 2026-02-01 -> 2026-02-28 | contents=321,546
2026-03: daily rows=9,841,378 | 2026-03-01 -> 2026-03-31 | contents=331,437
2026-04: daily rows=10,424,730 | 2026-04-01 -> 2026-04-30 | contents=362,172
2026-05: daily rows=11,687,376 | 2026-05-01 -> 2026-05-31 | contents=389,153

Before eligibility: 380,147 | Eligible rows: 16,513 | Clients: 36
Base rate (decline in May): 0.416 | Not seen in May (counted as decline): 0
Largest client share of rows: 26.1% | Top-5 clients: 71.9%
Per-client base rate spread (min / median / max, clients with >=30 rows):
  0.03 / 0.38 / 0.75  over 17 clients
Missing in features (share):
impressions_total      0.0
clicks_total           0.0
impressions_apr        0.0
clicks_apr             0.0
clicks_feb             0.0
momentum_apr_vs_feb    0.0
ctr                    0.0
weighted_position      0.0
active_days            0.0

Saved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)


## 1. Method choice and why

**Lane and question.** Refresh / Content Opportunity Scoring. The decision is *which pages should a reviewer open first this month?* That is a **ranking** question with a yes/no observed label (`y_decline_may`: May clicks below 80% of April clicks). So every method must produce a **score**, and the score is judged at **precision@50**, because 50 pages is the review budget. The label is an observed proxy, not proof that a refresh would help.

**The candidates, in order from simple to flexible:**

- **Frozen rule (ML-07).** April clicks at least 2x the Feb-Mar average, with at least 10 clicks in Feb and in Mar. It is the baseline every model has to beat, and it is recomputed in this notebook so it sits in the same run.
- **Logistic Regression.** The readable first model: one weight per feature. Counts are log-transformed and everything is scaled.
- **Decision Tree (depth 4).** Shallow on purpose, so the logic can be printed and read. The cost is many pages sharing the same score, so the tie policy matters.
- **Random Forest (the primary model, declared before any result).** Many trees voting. It can learn combinations a straight line cannot, for example "a surge only matters on pages that already had traffic".
- **Hist Gradient Boosting.** Small fixed settings, included only to check whether extra complexity earns anything.

**Not used:** K-Means and PCA (there is no archetype question here), and correlation tables (that was the ML-07 signal check).

**Rules I set before running anything.** Features are the 9 contract features and nothing is added. Hyperparameters are fixed in the code and are never tuned on a held-out fold. Complexity gets no credit by default: if a simpler method is about as good, the simpler one is the one to explain. The rows are the same 15,680 pages as the baseline (the 833 identical rows from one client are excluded, as in ML-07).

In [2]:
# 1) Analysis frame (same rows as the ML-07 baseline) + the candidates, all declared BEFORE any fold is scored
import json, platform
import numpy as np, pandas as pd, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

SEED, K_MAIN, PRIMARY = 42, 50, "Random Forest"          # primary model is declared here, before any result exists
KEYS, LABEL = ["client_hash_id", "content_hash_id"], "y_decline_may"
FEATURES = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb",
            "momentum_apr_vs_feb", "ctr", "weighted_position", "active_days"]          # the 9 contract features, nothing added
MIN_RATIO, HISTORY_FLOOR, BLOCK_COPIES = 2.0, 10, 20                                    # frozen ML-07 rule settings

df = pd.read_parquet(FRAME).reset_index(drop=True)
df["clicks_mar"] = (df["clicks_total"] - df["clicks_apr"] - df["clicks_feb"]).clip(lower=0)
prior = (df["clicks_feb"] + df["clicks_mar"]) / 2
df["ratio_apr"] = df["clicks_apr"] / prior.where(prior > 0)
VEC = ["clicks_feb", "clicks_mar", "clicks_apr", "impressions_apr", "weighted_position", "active_days"]
block = df.groupby(VEC, dropna=False)["content_hash_id"].transform("size") >= BLOCK_COPIES      # identical rows = not independent evidence
n_block = int(block.sum())
df = df[~block].reset_index(drop=True)                                                          # same analysis frame as ML-07
df["_tb"] = pd.util.hash_pandas_object(df["content_hash_id"].astype(str) + f"|{SEED}", index=False).to_numpy()   # same tie policy as ML-07

# The frozen ML-07 rule, recomputed here so the baseline sits in the same run as the models
flag = (df["ratio_apr"] >= MIN_RATIO) & (df["clicks_feb"] >= HISTORY_FLOOR) & (df["clicks_mar"] >= HISTORY_FLOOR)
strength = (np.log2(df["ratio_apr"].clip(lower=1)) / 3).clip(0, 1)
df["rule_score"] = np.where(flag, 100 * strength * df["impressions_apr"].rank(pct=True), 0.0).round(4)

assert "clicks_may" not in FEATURES and LABEL not in FEATURES and set(KEYS).isdisjoint(FEATURES), "label or ids in the features"
COUNTS = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb"]
def make_prep():       # logistic regression needs scaled inputs; counts are heavy-tailed, so log1p first
    return ColumnTransformer([("counts", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), COUNTS),
                              ("others", StandardScaler(), [c for c in FEATURES if c not in COUNTS])])
def make_models():     # fixed settings, no tuning on any held-out fold
    return {"Logistic Regression": make_pipeline(make_prep(), LogisticRegression(max_iter=2000)),
            "Decision Tree (depth 4)": DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=SEED),
            "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED),
            "Hist Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=SEED)}
RULE = "Frozen rule (ML-07)"
METHODS = [RULE, *make_models()]

def prec_at_k(y_, s_, tb_, k):          # sort by score, break exact ties with the seeded hash, take the top k
    o = np.lexsort((tb_, -s_)); return float(y_[o][:k].mean())
y_all, tb_all = df[LABEL].to_numpy(), df["_tb"].to_numpy()
print(f"Analysis frame: {len(df):,} rows | {df['client_hash_id'].nunique()} clients | base rate {y_all.mean():.3f} | excluded identical block: {n_block:,} rows")
print(f"Frozen rule over the whole frame: precision@{K_MAIN} = {prec_at_k(y_all, df['rule_score'].to_numpy(), tb_all, K_MAIN):.2f}  (ML-07 reported 0.42)")
print(f"Candidates: {METHODS} | primary (declared in advance): {PRIMARY} | seed {SEED}")
print(f"Versions: python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

Analysis frame: 15,680 rows | 36 clients | base rate 0.385 | excluded identical block: 833 rows
Frozen rule over the whole frame: precision@50 = 0.42  (ML-07 reported 0.42)
Candidates: ['Frozen rule (ML-07)', 'Logistic Regression', 'Decision Tree (depth 4)', 'Random Forest', 'Hist Gradient Boosting'] | primary (declared in advance): Random Forest | seed 42
Versions: python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1


## 2. Split design

**Five-fold GroupKFold on `client_hash_id`.** Every client sits in exactly one test fold, and the code asserts that no client appears on both sides.

**Why this is the honest split for this question.** The question is whether the ranking works for a client it has **not seen**. Clients are very different: the largest holds about a quarter of the rows, and client base rates range from 3% to 75%. With random rows, a model can recognize the client from its traffic size and position and read the answer from the client's name. That is memorizing, not learning. Section 3 runs the same Random Forest on random rows as a contrast.

**What stays identical for every method, the rule included:** the same rows, the same five folds, the same metric (precision@50 per fold), the same K, and the same tie policy (a seeded hash, never file order). The base rate of each fold is printed next to the results, and all five folds are shown, not only the average.

**What this split does not test.** It is one decision point (1 May 2026), so it says nothing about future months. Each fold is a different set of unseen clients with a different base rate, so some folds are easier than others. A time-forward check belongs to the validation audit (ML-09).

**What GroupKFold did in practice.** Because the portfolio is lopsided, fold 1 holds only the largest client (4,308 rows) and fold 2 holds only the second largest (2,907 rows). Those two folds each measure *one* client. Folds 3 to 5 hold 7, 15 and 12 clients.


In [3]:
# 2) Split design: 5-fold GroupKFold on client_hash_id, the same folds for every method (rule included)
X, groups = df[FEATURES], df["client_hash_id"].to_numpy()
folds = list(GroupKFold(n_splits=5).split(X, y_all, groups))
rows = []
for i, (tr, te) in enumerate(folds, 1):
    assert set(groups[tr]).isdisjoint(groups[te]), "a client appears on both sides of a fold"
    rows.append({"fold": i, "train_rows": len(tr), "test_rows": len(te), "test_clients": len(set(groups[te])),
                 "test_base_rate": round(float(y_all[te].mean()), 3),
                 "largest_client_share_in_test": round(float(pd.Series(groups[te]).value_counts(normalize=True).iloc[0]), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nAsserted: no client is in both train and test, in any fold. Metric: precision@50 per fold (K = the review budget); precision@10, @100 and AUC are secondary.")

 fold  train_rows  test_rows  test_clients  test_base_rate  largest_client_share_in_test
    1       11372       4308             1           0.331                         1.000
    2       12773       2907             1           0.343                         1.000
    3       12858       2822             7           0.396                         0.668
    4       12858       2822            15           0.366                         0.499
    5       12859       2821            12           0.521                         0.409

Asserted: no client is in both train and test, in any fold. Metric: precision@50 per fold (K = the review budget); precision@10, @100 and AUC are secondary.


## 3. Train + compare vs my baseline

**Model-vs-baseline table.** Five client-grouped folds, mean +/- sd across folds. The base rate is **0.391**, which is what random picking gives. Every row uses the same pages, the same folds, the same metric, the same K and the same tie policy.

| Method | P@10 | P@50 | P@100 | AUC | P@50 minus rule | Folds better than the rule |
|---|---|---|---|---|---|---|
| Frozen rule (ML-07) | 0.42 +/- 0.16 | 0.36 +/- 0.09 | 0.41 +/- 0.08 | 0.501 | - | - |
| **Logistic Regression** | **0.68 +/- 0.11** | **0.62 +/- 0.08** | **0.58 +/- 0.06** | **0.581** | **+0.25** | **5 of 5** |
| Decision Tree (depth 4) | 0.48 +/- 0.13 | 0.44 +/- 0.09 | 0.43 +/- 0.08 | 0.543 | +0.08 | 4 of 5 |
| Random Forest (declared primary) | 0.48 +/- 0.22 | 0.50 +/- 0.16 | 0.48 +/- 0.12 | 0.571 | +0.14 | 4 of 5 |
| Hist Gradient Boosting | 0.58 +/- 0.13 | 0.53 +/- 0.07 | 0.46 +/- 0.08 | 0.568 | +0.16 | 5 of 5 |

**What the table shows (observed, directional).**

- **The baseline is close to random on unseen clients.** The frozen rule scores 0.36 at precision@50 against a 0.39 base rate, and its AUC is 0.501. This agrees with the honest reading in ML-07. The printed line above also shows that in at least one fold the rule flags fewer than 50 pages (2,788 pages are tied at its cutoff), so part of its top-50 is filled in hash order. Comparing with the base rate gives the same story as comparing with the rule.
- **The simplest learned model wins at every K.** Logistic Regression is best at precision@10, @50 and @100 and has the best AUC. It is above the rule and above the fold's base rate in all five folds (by roughly 16 to 32 points over the base rate at precision@50). It is also higher than Hist Gradient Boosting in all five folds and higher than the Random Forest in four of five.
- **Complexity was not rewarded.** The Random Forest I declared as primary is the least stable method: 0.50 +/- 0.16, and in fold 2 it scores 0.22 against a base rate of 0.34, which is worse than random. Gradient boosting is steadier but still below Logistic Regression everywhere. The depth-4 tree has up to 1,090 pages tied at its cutoff, so its top-50 depends on the tie rule.
- **The ranking is weak overall and useful mostly at the head.** All AUCs are about 0.57 to 0.58 (the rule is 0.50), while precision at the top of the list is much higher than the base rate. That is where the 50-page review budget lives.
- **The split matters as much as the model.** The same Random Forest scores **0.73** at precision@50 with random rows and **0.50** with client-grouped folds. Only the split changed. Random rows would have let the model recognize clients it had already seen.

**Honest limits.** Folds 1 and 2 are single clients, so each is one client's behavior. Fold base rates run from 0.33 to 0.52. A 50-page list moves by roughly +/-13 points by chance alone, and the sd across folds is about 0.08, so the **gap to the baseline** is the finding, not the exact decimals. I picked Logistic Regression *after* seeing five folds, which adds some optimism, so the validation audit (ML-09) will test it on a different window. Tree-ensemble numbers can shift a few points between library versions. This run used scikit-learn 1.6.1 with fixed seeds (42).

In [4]:
# 3) Train on each fold's training clients, score the held-out clients, compare with the frozen rule
res, oof, rf_models = [], {n: np.full(len(df), np.nan) for n in METHODS}, []
for i, (tr, te) in enumerate(folds, 1):
    scores = {RULE: df["rule_score"].to_numpy()[te]}
    for name, m in make_models().items():
        m.fit(X.iloc[tr], y_all[tr]); scores[name] = m.predict_proba(X.iloc[te])[:, 1]
        if name == PRIMARY: rf_models.append(m)
    for name, s in scores.items():
        oof[name][te] = s
        ties = int((s == np.sort(s)[::-1][K_MAIN - 1]).sum())            # rows sharing the exact score at the top-50 cutoff
        res.append({"fold": i, "method": name, "base_rate": float(y_all[te].mean()), "ties_at_cutoff": ties,
                    "p10": prec_at_k(y_all[te], s, tb_all[te], 10), "p50": prec_at_k(y_all[te], s, tb_all[te], 50),
                    "p100": prec_at_k(y_all[te], s, tb_all[te], 100), "auc": roc_auc_score(y_all[te], s)})
res = pd.DataFrame(res)

p50 = res.pivot(index="fold", columns="method", values="p50")[METHODS]
p50.insert(0, "base_rate", res.groupby("fold")["base_rate"].first())
print("precision@50 in each held-out fold (the folds are different sets of unseen clients):"); print(p50.round(2).to_string())

summ = []
for name in METHODS:
    r = res[res["method"] == name]
    d = p50[name] - p50[RULE]
    summ.append({"method": name, "P@10": f"{r.p10.mean():.2f} +/- {r.p10.std():.2f}", "P@50": f"{r.p50.mean():.2f} +/- {r.p50.std():.2f}",
                 "P@100": f"{r.p100.mean():.2f} +/- {r.p100.std():.2f}", "AUC": f"{r.auc.mean():.3f}",
                 "P@50 minus rule": "-" if name == RULE else f"{d.mean():+.2f}", "folds better than rule": "-" if name == RULE else f"{int((d > 0).sum())} of 5",
                 "max ties at cutoff": int(r.ties_at_cutoff.max())})
print(f"\nSummary over 5 folds (mean +/- sd across folds). Base rate: {res.groupby('fold')['base_rate'].first().mean():.3f} = what random picking gives.")
print(pd.DataFrame(summ).to_string(index=False))

fl = [int((df["rule_score"].to_numpy()[te] > 0).sum()) for tr, te in folds]
fp = [round(float(y_all[te][df["rule_score"].to_numpy()[te] > 0].mean()), 2) if n else None for (tr, te), n in zip(folds, fl)]
print(f"\nPages the frozen rule flags in each fold: {fl} | decline rate among them: {fp} (fewer than 50 flagged means its top-50 is filled with unflagged pages in hash order)")

rs = []                                                   # the "easy exam": same model, random rows (clients on both sides)
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y_all):
    m = make_models()[PRIMARY]; m.fit(X.iloc[tr], y_all[tr]); rs.append(prec_at_k(y_all[te], m.predict_proba(X.iloc[te])[:, 1], tb_all[te], K_MAIN))
print(f"\n{PRIMARY} precision@50 with RANDOM rows: {np.mean(rs):.2f} (folds {np.round(rs, 2)}) vs client-grouped: {p50[PRIMARY].mean():.2f}. Same model, same features: only the split changed.")

precision@50 in each held-out fold (the folds are different sets of unseen clients):
method  base_rate  Frozen rule (ML-07)  Logistic Regression  Decision Tree (depth 4)  Random Forest  Hist Gradient Boosting
fold                                                                                                                       
1            0.33                 0.24                 0.56                     0.38           0.60                    0.54
2            0.34                 0.32                 0.56                     0.32           0.22                    0.44
3            0.40                 0.42                 0.72                     0.50           0.62                    0.62
4            0.37                 0.38                 0.56                     0.46           0.54                    0.50
5            0.52                 0.46                 0.68                     0.54           0.54                    0.54

Summary over 5 folds (mean +/- sd across folds

## 4. Errors and interpretation

**What the models lean on.** For the Random Forest, shuffling `active_days` and `impressions_apr` hurts AUC the most (mean drop 0.040 and 0.029), and both are in the top three in all five folds. `momentum_apr_vs_feb` follows (0.016, top three in 3 of 5). Everything else is near zero, including `clicks_apr`, even though the label is built from April clicks. The spread between folds is as large as the means (for example 0.040 +/- 0.027), so I claim only the *group* of leading features, not their order. For the Logistic Regression I carry forward, the coefficient table above gives the direction. Several count features overlap (total, April, February), so I read a sign only when it agrees across folds. Nothing looks suspiciously perfect: the biggest AUC drop is 0.04, and the best single feature had AUC 0.549 in ML-04. Plausible reasons, not tested: `active_days` separates steady pages from bursty ones, and `impressions_apr` together with small clicks marks pages whose click counts are noisy.

**What the errors look like (Random Forest, out-of-fold).**

- **Hits and misses look the same.** Across the five top-50 lists, 126 of 250 pages declined (50%, against a 39% base rate). But the median hit and miss are almost identical: 14 April clicks, 2 February clicks, 71 active days, and a ratio of about 2.4. Only the average position differs a little (4.9 for hits, 6.7 for misses).
- **The top of the list is full of tiny pages.** The median page there has only 14 clicks in April, and the eligibility floor is 10. Counts that small are noisy, and the filter keeps pages whose April cleared the floor. A fall below 80% of April can then happen by chance (regression to the mean). So part of what the models find may be "April was a lucky month for a small page" and not real decay. This is a hypothesis I have not tested yet. The sensitivity checks in ML-09 (for example, removing click-level features and splitting by April click size) will test it.
- **Three confident misses.** All three come from fold 2, which is a single client, and are ranked 2 to 4 with scores around 0.73. They have 0 clicks in February, 6 to 7 in March and 11 to 13 in April, and 12, 24 and 13 clicks in May, at positions 31 to 41. They are hard because the counts are tiny: for a page with 11 April clicks the "decline" bar is 8.8 clicks, so one or two clicks decide the label. They also look like new pages that are still growing.
- **Declines the model misses.** 1,205 of 6,043 declined pages (20%) score in the bottom quarter of their fold, and 1,814 (30%) score in the top quarter, while chance would give 25% and 25%. The ordering is weak. The missed declines are bigger and steadier pages: a median of 31 April clicks (against 16 for the declines it catches), 20 February clicks (against 5), 89 active days (against 80), and a flat ratio of 1.06 (against 1.79). The model catches small rising pages that fall back and misses established flat pages that decline, and the second kind is closer to the content decay a refresh queue is meant to find.

**Bottom line (observed, directional, decision-support).** On unseen clients, a simple Logistic Regression ranks the head of the queue clearly better than the ML-07 rule and than random picking. The model is still weak overall (AUC about 0.58), its picks lean toward small noisy pages, and it misses decline on established pages. It should be treated as a way to order a human review, not as a detector of decay.

*The depth-3 tree printed above was fit on all rows for reading only and was not scored.*

In [5]:
# 4) Errors and interpretation (primary model's out-of-fold scores: every row scored by a model that never saw its client)
cols = ["clicks_apr", "clicks_feb", "impressions_apr", "weighted_position", "active_days", "ratio_apr"]
oof_rf = oof[PRIMARY]; d = df.assign(oof=oof_rf)

# (a1) The simplest learned model: standardized logistic coefficients in every fold (sign = direction, size = weight; log-scaled counts overlap, so read signs only where folds agree)
names = COUNTS + [c for c in FEATURES if c not in COUNTS]          # the order ColumnTransformer outputs
coefs = pd.DataFrame([pd.Series(make_models()["Logistic Regression"].fit(X.iloc[tr], y_all[tr])[-1].coef_[0], index=names) for tr, te in folds])
coef_tab = pd.DataFrame({"mean_coef": coefs.mean(), "sd_across_folds": coefs.std(),
                         "same_sign_folds": (np.sign(coefs) == np.sign(coefs.mean())).sum()}).sort_values("mean_coef", key=abs, ascending=False)
print("Logistic Regression coefficients (standardized inputs, mean over 5 folds):"); print(coef_tab.round(3).to_string())

# (a2) What does the forest lean on? Permutation importance on each held-out fold; the spread across folds matters as much as the mean
imp = pd.DataFrame([permutation_importance(m, X.iloc[te], y_all[te], scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1).importances_mean
                    for (tr, te), m in zip(folds, rf_models)], columns=FEATURES)
imp_tab = pd.DataFrame({"mean_AUC_drop": imp.mean(), "sd_across_folds": imp.std(),
                        "folds_in_top3": (imp.rank(axis=1, ascending=False) <= 3).sum()}).sort_values("mean_AUC_drop", ascending=False)
print("\nForest permutation importance (drop in AUC when the column is shuffled):"); print(imp_tab.round(3).to_string())

# (b) Top-50 of every fold: hits (declined) vs misses (did not decline)
tops = []
for i, (tr, te) in enumerate(folds, 1):
    o = np.lexsort((tb_all[te], -oof_rf[te]))[:K_MAIN]
    tops.append(d.iloc[te[o]].assign(fold=i, rank=np.arange(1, K_MAIN + 1)))
tops = pd.concat(tops)
print(f"\nTop-{K_MAIN} lists of the 5 folds together: {int(tops[LABEL].sum())} hits, {int((1 - tops[LABEL]).sum())} misses. Median feature values:")
print(tops.groupby(LABEL)[cols].median().rename(index={0: "miss (did not decline)", 1: "hit (declined)"}).round(2).to_string())
print("\nThree confident misses (highest scores that did NOT decline; clicks_may is shown for evaluation only):")
print(tops[tops[LABEL] == 0].sort_values("oof", ascending=False).head(3)[["fold", "rank", "oof", "clicks_feb", "clicks_mar", "clicks_apr", "clicks_may", "impressions_apr", "weighted_position", "active_days"]].round(2).to_string(index=False))

# (c) Declines the model missed: pages that declined but scored low inside their fold
d["pct_in_fold"] = np.nan
for tr, te in folds: d.loc[te, "pct_in_fold"] = pd.Series(oof_rf[te]).rank(pct=True).to_numpy()
dec = d[d[LABEL] == 1]; low, high = dec[dec["pct_in_fold"] < .25], dec[dec["pct_in_fold"] >= .75]
print(f"\nDeclined pages scored in the bottom quarter of their fold: {len(low):,} of {len(dec):,} ({len(low) / len(dec):.0%}); in the top quarter: {len(high):,} ({len(high) / len(dec):.0%})")
print(pd.DataFrame({"declined, low score": low[cols].median(), "declined, high score": high[cols].median()}).round(2).to_string())

# (d) A shallow tree fit on all rows, for reading only (not scored)
print("\nDepth-3 tree for reading only (fit on all rows):"); print(export_text(DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=SEED).fit(X, y_all), feature_names=FEATURES))

json.dump({"seed": SEED, "primary_declared_in_advance": PRIMARY, "k": K_MAIN, "rows": int(len(df)), "excluded_identical_block_rows": n_block,
           "split": "5-fold GroupKFold on client_hash_id", "features": FEATURES,
           "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__},
           "models": {n: str(m.get_params(deep=False)) for n, m in make_models().items()},
           "per_fold": res.round(4).to_dict("records"), "random_rows_rf_p50": [round(float(v), 4) for v in rs],
           "logistic_coefficients": coef_tab.round(4).reset_index().rename(columns={"index": "feature"}).to_dict("records"),
           "permutation_importance": imp_tab.round(4).reset_index().rename(columns={"index": "feature"}).to_dict("records")},
          open("work/outputs/w05_model_metrics.json", "w"), indent=2, default=str)
print("Saved: work/outputs/w05_model_metrics.json (commit this one)")

Logistic Regression coefficients (standardized inputs, mean over 5 folds):
                     mean_coef  sd_across_folds  same_sign_folds
impressions_total        1.274            0.155                5
impressions_apr         -1.219            0.130                5
clicks_total            -0.354            0.146                5
momentum_apr_vs_feb      0.286            0.020                5
clicks_apr               0.286            0.062                5
clicks_feb              -0.127            0.043                5
active_days              0.112            0.098                4
ctr                     -0.070            0.024                5
weighted_position        0.029            0.115                3

Forest permutation importance (drop in AUC when the column is shuffled):
                     mean_AUC_drop  sd_across_folds  folds_in_top3
active_days                  0.040            0.027              5
impressions_apr              0.029            0.014              5


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.